# Лекция 6. Файлы, модули, CSV и JSON


## 1. Файлы и пути

Файл хранит данные на диске и сохраняется после завершения программы.

**Относительный путь** считается от текущей рабочей папки:

```text
data/students.txt
```

**Абсолютный путь** содержит полный путь к файлу.

В Jupyter Notebook текущая рабочая папка может отличаться от папки, в которой лежит сам ноутбук.


In [1]:
import os

print("Текущая рабочая папка:")
print(os.getcwd())


Текущая рабочая папка:
/Users/demon/Desktop/PythonCourse/PythonCourse_2026/06/Лекция


Для примеров создадим небольшой текстовый файл.


In [3]:
file = open("text_sample.txt", "w", encoding="utf-8")
file.write("Первая строка\n")
file.write("Вторая строка\n")
file.write("Третья строка\n")
file.close()


## 2. Функция `open()`

Сначала рассмотрим работу с файлами **без** контекстного менеджера.

Общий вид:

```python
file = open("имя_файла", "режим", encoding="utf-8")
```

`open()` возвращает файловый объект.

После работы файл необходимо закрыть:

```python
file.close()
```


### 2.1. Основные режимы открытия файла

| Режим | Назначение |
|---|---|
| `"r"` | чтение; файл должен существовать |
| `"w"` | запись; старое содержимое очищается |
| `"a"` | добавление в конец |
| `"x"` | создать новый файл; ошибка, если он уже существует |
| `"r+"` | чтение и запись без автоматического очищения |
| `"w+"` | чтение и запись с очищением файла |
| `"a+"` | чтение и добавление в конец |
| `"rb"` | чтение бинарного файла |
| `"wb"` | запись бинарного файла |

Символ `b` означает **binary** — бинарный режим.

> Запись `"rw"` не является корректным режимом `open()`. Для чтения и записи используют, например, `"r+"`, `"w+"` или `"a+"`.


### 2.2. Чтение всего файла: `read()`


In [4]:
file = open("text_sample.txt", "r", encoding="utf-8")

text = file.read()
print(text)

file.close()


Первая строка
Вторая строка
Третья строка



### 2.3. Чтение одной строки: `readline()`


In [5]:
file = open("text_sample.txt", "r", encoding="utf-8")

line = file.readline()
print(repr(line))

file.close()


'Первая строка\n'


### 2.4. Чтение всех строк: `readlines()`

`readlines()` возвращает список строк.


In [6]:
file = open("text_sample.txt", "r", encoding="utf-8")

lines = file.readlines()
file.close()

print(lines)


['Первая строка\n', 'Вторая строка\n', 'Третья строка\n']


### 2.5. Перебор файла циклом

Для большого файла обычно лучше не загружать его целиком в память.


In [7]:
file = open("text_sample.txt", "r", encoding="utf-8")

for line in file:
    print(line.rstrip("\n"))

file.close()


Первая строка
Вторая строка
Третья строка


### 2.6. Запись: `write()` и `writelines()`

`write()` записывает одну строку.

Важно: `write()` сам не добавляет `\n`.


In [8]:
file = open("text_sample_write.txt", "w", encoding="utf-8")

file.write("MIPT\n")
file.write("Python\n")

file.close()


In [9]:
file = open("many_lines.txt", "w", encoding="utf-8")

file.writelines(["one\n", "two\n", "three\n"])

file.close()


### 2.7. Добавление в конец: режим `"a"`


In [10]:
file = open("text_sample_write.txt", "a", encoding="utf-8")
file.write("Новая строка\n")
file.close()


### 2.8. Текущая позиция: `tell()` и `seek()`

Файл имеет текущую позицию чтения/записи.

- `file.tell()` — вернуть текущую позицию;
- `file.seek(position)` — перейти к указанной позиции.

Для текстовых файлов чаще всего безопасно использовать `seek(0)`, чтобы вернуться в начало файла.


In [11]:
file = open("text_sample.txt", "r", encoding="utf-8")

print("Начальная позиция:", file.tell())

part = file.read(5)
print("Прочитали:", repr(part))
print("Новая позиция:", file.tell())

file.seek(0)
print("После seek(0):", file.tell())
print(file.readline())

file.close()


Начальная позиция: 0
Прочитали: 'Перва'
Новая позиция: 10
После seek(0): 0
Первая строка



После полного `read()` указатель находится в конце. Поэтому повторный `read()` вернёт пустую строку, пока мы не изменим позицию.


In [ ]:
file = open("text_sample.txt", "r", encoding="utf-8")

print(repr(file.read()))
print(repr(file.read()))

file.seek(0)
print(repr(file.readline()))

file.close()


### 2.9. Проверка состояния файла

Свойство `file.closed` показывает, закрыт ли файл.


In [12]:
file = open("text_sample.txt", encoding="utf-8")
print(file.closed)

file.close()
print(file.closed)


False
True


### 2.10. Возможные ошибки


In [17]:
try:
    file = open("missing_file.txt", "r", encoding="utf-8")
    print(file.read())
    file.close()
except FileNotFoundError:
    print("Файл не найден")
    file.close()

Файл не найден


## 3. Контекстный менеджер `with`

При обычной работе через `open()` легко забыть вызвать `close()`.

Контекстный менеджер:

```python
with open(...) as file:
    ...
```

автоматически закрывает файл после выхода из блока.


In [18]:
with open("text_sample.txt", "r", encoding="utf-8") as file:
    print(file.read())
    print("Внутри блока:", file.closed)

print("После блока:", file.closed)


Первая строка
Вторая строка
Третья строка

Внутри блока: False
После блока: True


Сравните:

```python
file = open("text_sample.txt", encoding="utf-8")
text = file.read()
file.close()
```

и

```python
with open("text_sample.txt", encoding="utf-8") as file:
    text = file.read()
```

В реальном Python-коде второй вариант обычно предпочтительнее.


## 4. Текстовый и бинарный режим

TXT, CSV и JSON обычно открывают как текст:

```python
open("file.txt", "r", encoding="utf-8")
```

Картинки, архивы и другие бинарные данные читают через `"rb"` и `"wb"`.


In [21]:
with open("text_sample.txt", "rb") as file:
    data = file.read(20)

print(data)
print(type(data))


b'\xd0\x9f\xd0\xb5\xd1\x80\xd0\xb2\xd0\xb0\xd1\x8f \xd1\x81\xd1\x82\xd1\x80\xd0'
<class 'bytes'>


### UTF-8, `encode()`, `decode()`, `ord()` и `chr()`

Строка Python состоит из Unicode-символов.

- `ord(symbol)` — код символа;
- `chr(code)` — символ по коду;
- `text.encode("utf-8")` — строка → байты;
- `data.decode("utf-8")` — байты → строка.


In [22]:
text = "Привет"

print(ord("П"))
print(chr(1055))

data = text.encode("utf-8")
print(data)
print(data.decode("utf-8"))


1055
П
b'\xd0\x9f\xd1\x80\xd0\xb8\xd0\xb2\xd0\xb5\xd1\x82'
Привет


## 5. CSV

CSV — текстовый формат для таблиц.

Наивно разделять строки через `split(",")` опасно: внутри поля тоже может встретиться запятая. Поэтому используется модуль `csv`.


### 5.1. Сначала посмотрим CSV как обычный текстовый файл


In [55]:
with open("csv_sample.csv", "w", encoding="utf-8", newline="") as file:
    file.write("name,established_year,city\n")
    file.write("MIPT,1946,Dolgoprudny\n")
    file.write("MSU,1755,Moscow\n")
    file.write("HSE,1992,Moscow\n")

with open("csv_sample.csv", encoding="utf-8") as file:
    lines = file.readlines()

names = lines[0]
table = lines[1:]

print("Заголовок:", names)
print("Строки:", table)


Заголовок: name,established_year,city

Строки: ['MIPT,1946,Dolgoprudny\n', 'MSU,1755,Moscow\n', 'HSE,1992,Moscow\n']


### 5.2. Основные функции и классы `csv`

- `csv.reader(file)` — читать строки как списки;
- `csv.writer(file)` — записывать списки;
- `csv.DictReader(file)` — читать строки как словари;
- `csv.DictWriter(file, fieldnames=...)` — записывать словари;
- `writer.writerow(row)` — записать одну строку;
- `writer.writerows(rows)` — записать несколько строк;
- `DictWriter.writeheader()` — записать заголовок;
- `delimiter=";"` — задать другой разделитель.


### 5.3. `csv.reader`


In [56]:
import csv

with open("csv_sample.csv", encoding="utf-8", newline="") as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)


['name', 'established_year', 'city']
['MIPT', '1946', 'Dolgoprudny']
['MSU', '1755', 'Moscow']
['HSE', '1992', 'Moscow']


Все значения из CSV читаются как строки.


### 5.4. `csv.DictReader`

Первая строка файла становится названиями ключей.


In [57]:
import csv

with open("csv_sample.csv", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row)


{'name': 'MIPT', 'established_year': '1946', 'city': 'Dolgoprudny'}
{'name': 'MSU', 'established_year': '1755', 'city': 'Moscow'}
{'name': 'HSE', 'established_year': '1992', 'city': 'Moscow'}


Пример обработки числового столбца:


In [58]:
import csv

with open("csv_sample.csv", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)

    total = 0
    count = 0

    for row in reader:
        total += int(row["established_year"])
        count += 1

print(total / count)


1897.6666666666667


### 5.5. `csv.writer`


In [59]:
import csv

with open("csv_sample_write.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerows([
        ["a", "b", 1],
        [1, 2, 3],
        [4, 5, 6],
    ])


### 5.6. `csv.DictWriter`


In [ ]:
import csv

with open("people.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["name", "age"])
    writer.writeheader()
    writer.writerow({"name": "Ivan", "age": 10})
    writer.writerow({"name": "Anna", "age": 12})


### 5.7. Другой разделитель


In [ ]:
import csv

with open("semicolon.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file, delimiter=";")
    writer.writerow(["city", "population"])
    writer.writerow(["Moscow", 13000000])


## 6. JSON

JSON хранит вложенные структуры данных.

Соответствие Python:

| JSON | Python |
|---|---|
| object | `dict` |
| array | `list` |
| string | `str` |
| number | `int`, `float` |
| true / false | `True` / `False` |
| null | `None` |


### 6.1. Основные функции `json`

- `json.load(file)` — JSON-файл → объект Python;
- `json.loads(text)` — JSON-строка → объект Python;
- `json.dump(obj, file)` — объект Python → JSON-файл;
- `json.dumps(obj)` — объект Python → JSON-строка.

Полезные параметры:
- `ensure_ascii=False` — сохранять кириллицу нормально;
- `indent=2` или `indent=4` — красивое форматирование;
- `sort_keys=True` — сортировать ключи.


In [60]:
import json

data = {
    "Nikita": {
        "age": 19,
        "pro_progger": True,
        "student": True
    },
    "Dima": {
        "age": 20,
        "pro_progger": False,
        "student": True
    }
}

with open("json_sample.json", "w", encoding="utf-8") as file:
    json.dump(data, file, ensure_ascii=False, indent=2)


### 6.2. `json.load()`


In [61]:
import json

with open("json_sample.json", encoding="utf-8") as file:
    data = json.load(file)

for key, value in data.items():
    print(key, value)


Nikita {'age': 19, 'pro_progger': True, 'student': True}
Dima {'age': 20, 'pro_progger': False, 'student': True}


### 6.3. `json.dumps()` и `json.loads()`


In [ ]:
import json

person = {"name": "Иван", "age": 30}

text = json.dumps(person, ensure_ascii=False)
print(text)

restored = json.loads(text)
print(restored)


## 7. Модули Python

**Модуль** — файл `.py`, содержащий Python-код: функции, классы, переменные и другой код.

Импорт позволяет использовать код из другого модуля.


### 7.1. `import module`

```python
import math
```

После этого обращаемся через имя модуля:

```python
math.sqrt(25)
math.pi
```


In [ ]:
import math

print(math.sqrt(25))
print(math.pi)


### 7.2. `from module import name`

Можно импортировать конкретные объекты:


In [ ]:
from math import sqrt, pi

print(sqrt(25))
print(pi)


### 7.3. Псевдоним `as`


In [ ]:
from math import factorial as fact

print(fact(5))


### 7.4. `from module import *`

```python
from math import *
```

Так импортируются почти все публичные имена модуля прямо в текущую область имён.

Этот вариант **лучше не использовать в обычном коде**, потому что:
- непонятно, откуда пришло имя;
- возможны конфликты имён;
- код сложнее читать и отлаживать.


In [ ]:
from math import *

print(sqrt(16))
print(pi)


### 7.5. Как Python ищет модули

Python ищет модуль:
1. среди встроенных модулей;
2. в текущем проекте;
3. в каталогах из `sys.path`;
4. среди установленных библиотек.

`pip install ...` устанавливает библиотеку, а `import ...` подключает её в программу.


## 8. Собственные модули и обычный проект/PyCharm

Пусть в одной папке находятся два файла:

```text
project/
├── main.py
└── geometry.py
```

`geometry.py`:

```python
def rectangle_area(a, b):
    return a * b
```

`main.py`:

```python
import geometry

print(geometry.rectangle_area(3, 4))
```

В PyCharm достаточно, чтобы оба файла находились в одном проекте/доступном Python-пути.

Можно импортировать и конкретную функцию:

```python
from geometry import rectangle_area
```


## 9. `__name__` и `__main__`

У каждого Python-модуля есть специальная переменная `__name__`.

Если файл **запущен напрямую**, то:

```python
__name__ == "__main__"
```

Если файл **импортирован**, то `__name__` содержит имя модуля.


In [ ]:
print(__name__)


Поэтому часто используют:

```python
if __name__ == "__main__":
    ...
```

Это означает:

> выполнять этот код только при прямом запуске файла.


In [ ]:
def rectangle_area(a, b):
    return a * b


if __name__ == "__main__":
    print(rectangle_area(3, 4))


### Частый шаблон программы


In [ ]:
def greet(name):
    return f"Привет, {name}!"


def main():
    name = input("Введите имя: ")
    print(greet(name))


if __name__ == "__main__":
    main()


## 10. Модуль `math`

### Основные функции и объекты

- `math.sqrt(x)` — квадратный корень;
- `math.pow(a, b)` — степень;
- `math.ceil(x)` — округление вверх;
- `math.floor(x)` — округление вниз;
- `math.trunc(x)` — отбросить дробную часть;
- `math.fabs(x)` — модуль;
- `math.factorial(n)` — факториал;
- `math.gcd(a, b)` — НОД;
- `math.lcm(a, b)` — НОК;
- `math.sin(x)`, `math.cos(x)`, `math.tan(x)` — тригонометрия;
- `math.radians(x)` — градусы → радианы;
- `math.degrees(x)` — радианы → градусы;
- `math.log(x)`, `math.log10(x)` — логарифмы;
- `math.exp(x)` — \(e^x\);
- `math.isclose(a, b)` — приблизительное равенство;
- `math.pi`, `math.e` — константы.


In [23]:
import math

print(math.sqrt(144))
print(math.factorial(5))
print(math.gcd(84, 30))
print(math.lcm(12, 18))
print(math.ceil(4.2))
print(math.floor(4.9))


12.0
120
6
36
5
4


In [ ]:
import math

r = 5
print("Длина окружности:", 2 * math.pi * r)
print("Площадь:", math.pi * r ** 2)

angle = 30
print("sin(30°):", math.sin(math.radians(angle)))


## 11. Модуль `random`

### Основные функции

- `random.randint(a, b)` — целое число от `a` до `b`;
- `random.randrange(...)` — случайное число из диапазона;
- `random.random()` — число от 0 до 1;
- `random.uniform(a, b)` — вещественное число из диапазона;
- `random.choice(sequence)` — случайный элемент;
- `random.choices(sequence, k=n)` — несколько элементов с повторениями;
- `random.sample(sequence, k=n)` — несколько разных элементов;
- `random.shuffle(list)` — перемешать список;
- `random.seed(x)` — задать начальное состояние генератора.


In [27]:
import random

print("Кубик:", random.randint(1, 6))
print("Монета:", random.choice(["Орёл", "Решка"]))

students = ["Аня", "Борис", "Вера", "Глеб", "Даша"]
print("Отвечает:", random.choice(students))
print("Команда:", random.sample(students, 3))


Кубик: 6
Монета: Решка
Отвечает: Аня
Команда: ['Вера', 'Борис', 'Даша']


In [42]:
import random

heads = 0

for _ in range(100):
    if random.choice(["Орёл", "Решка"]) == "Орёл":
        heads += 1

print("Орёл:", heads)
print("Относительная частота:", heads / 100)


Орёл: 60
Относительная частота: 0.6


## 12. Модуль `datetime`

### Основные классы и методы

- `datetime.datetime.now()` — текущие дата и время;
- `datetime.date.today()` — текущая дата;
- `datetime.datetime(...)` — создать дату и время;
- `datetime.date(...)` — создать дату;
- `datetime.time(...)` — создать время;
- `datetime.timedelta(...)` — промежуток времени;
- `.strftime(format)` — дата → строка;
- `datetime.datetime.strptime(text, format)` — строка → дата;
- `.weekday()` — день недели 0–6;
- `.timestamp()` — Unix-время;
- `datetime.datetime.fromtimestamp(x)` — из Unix-времени.

Полезные свойства: `.year`, `.month`, `.day`, `.hour`, `.minute`, `.second`.


In [45]:
import datetime

now = datetime.datetime.now()
today = datetime.date.today()

print(now)
print(now.strftime("%d.%m.%Y %H:%M"))


2026-10-17 12:17:15.751419
17.10.2026 12:17


In [46]:
import datetime

start = datetime.date(2026, 10, 1)
finish = datetime.date(2026, 10, 20)

print("Разница:", finish - start)
print("Через неделю:", start + datetime.timedelta(days=7))


Разница: 19 days, 0:00:00
Через неделю: 2026-10-08


In [52]:
start = datetime.datetime.now()
lst = []
for i in range(100000000):
    lst.append(i)


finish = datetime.datetime.now()
print(finish - start)

0:00:02.414758


## 13. Модуль `os`

`os` предоставляет функции для работы с операционной системой, каталогами, файлами и переменными окружения.

### Основные функции

- `os.getcwd()` — текущая папка;
- `os.chdir(path)` — сменить папку;
- `os.listdir(path)` — содержимое папки;
- `os.mkdir(path)` — создать папку;
- `os.makedirs(path, exist_ok=True)` — создать дерево папок;
- `os.remove(path)` — удалить файл;
- `os.rename(old, new)` — переименовать;
- `os.rmdir(path)` — удалить пустую папку;
- `os.path.join(...)` — собрать путь;
- `os.path.exists(path)` — существует ли путь;
- `os.path.isfile(path)` — файл ли это;
- `os.path.isdir(path)` — папка ли это;
- `os.path.basename(path)` — имя файла;
- `os.path.dirname(path)` — родительский путь;
- `os.environ` — переменные окружения;
- `os.getenv(name)` — получить переменную окружения.


In [53]:
import os

print(os.getcwd())
print(os.listdir(".")[:10])

path = os.path.join("data", "students.txt")
print(path)
print(os.path.exists(path))


/Users/demon/Desktop/PythonCourse/PythonCourse_2026/06/Лекция
['text_sample.txt', 'text_sample_write.txt', 'Python_lecture_files.ipynb', '.DS_Store', 'many_lines.txt', 'json_sample.json', 'examples', '.ipynb_checkpoints']
data/students.txt
False


Пример переименования:

```python
import os

os.rename("old_name.txt", "new_name.txt")
```

Перед выполнением полезно проверить, что исходный файл существует.


## 14. `pathlib.Path`

`pathlib` — современный объектно-ориентированный способ работы с путями.

### Основные методы и свойства

- `Path.cwd()` — текущая папка;
- `Path.home()` — домашняя папка;
- `Path("file.txt")` — объект пути;
- `.exists()` — существование;
- `.is_file()` — файл;
- `.is_dir()` — папка;
- `.mkdir()` — создать папку;
- `.iterdir()` — содержимое папки;
- `.glob("*.txt")` — поиск по маске;
- `.read_text()` — прочитать текст;
- `.write_text(text)` — записать текст;
- `.rename(...)` — переименовать;
- `.unlink()` — удалить файл;
- `.name` — имя файла;
- `.stem` — имя без расширения;
- `.suffix` — расширение;
- `.parent` — родительская папка;
- `.resolve()` — абсолютный путь.


In [ ]:
from pathlib import Path

folder = Path("data")
file_path = folder / "students.csv"

print(file_path)
print(file_path.name)
print(file_path.suffix)
print(file_path.parent)


## 15. Модуль `sys`

`sys` предоставляет информацию о Python-интерпретаторе и параметрах запуска программы.

### Основные объекты и функции

- `sys.version` — версия Python;
- `sys.executable` — путь к интерпретатору;
- `sys.path` — где Python ищет модули;
- `sys.argv` — аргументы командной строки;
- `sys.exit()` — завершить программу;
- `sys.platform` — платформа;
- `sys.stdin` — стандартный ввод;
- `sys.stdout` — стандартный вывод;
- `sys.stderr` — поток ошибок;
- `sys.getsizeof(obj)` — примерный размер объекта.


In [54]:
import sys

print(sys.version)
print(sys.executable)
print(sys.platform)
print(sys.argv)


3.12.2 (v3.12.2:6abddd9f6a, Feb  6 2024, 17:02:06) [Clang 13.0.0 (clang-1300.0.29.30)]
/Library/Frameworks/Python.framework/Versions/3.12/bin/python3.12
darwin
['/Users/demon/Library/Python/3.12/lib/python/site-packages/ipykernel_launcher.py', '-f', '/Users/demon/Library/Jupyter/runtime/kernel-b54abe63-146b-43b7-b08a-0d3777a145fc.json']


### 15.1. `sys.argv`

Если файл `main.py` запустить так:

```bash
python main.py Ivan 17
```

то:

```python
sys.argv
```

будет примерно:

```python
["main.py", "Ivan", "17"]
```

Первый элемент — имя запущенного файла.

Простой вариант обработки:


In [ ]:
# Код для обычного .py-файла:
#
# import sys
#
# name = sys.argv[1]
# age = int(sys.argv[2])
#
# print(name, age)


Недостаток `sys.argv`: программисту приходится самостоятельно проверять количество аргументов, преобразовывать типы и объяснять пользователю формат запуска.

Для серьёзных консольных программ удобнее `argparse`.


## 16. Модуль `argparse`

`argparse` помогает создавать интерфейс командной строки.

Он умеет:
- разбирать аргументы;
- преобразовывать их типы;
- задавать значения по умолчанию;
- делать аргументы обязательными или необязательными;
- автоматически создавать справку `--help`.


### 16.1. Минимальный пример

Файл `main.py`:

```python
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("name")

args = parser.parse_args()

print(args.name)
```

Запуск:

```bash
python main.py Ivan
```


### 16.2. Позиционные и именованные аргументы

Пример из исходных материалов лекции:

```python
import argparse

parser = argparse.ArgumentParser()

parser.add_argument("name", type=str, help="Name")
parser.add_argument(
    "--age",
    type=int,
    help="Age",
    default=32,
    required=False
)

args = parser.parse_args()
print(args)
```

Возможные запуски:

```bash
python main.py Ivan
python main.py Ivan --age 18
```

`name` — позиционный аргумент.

`--age` — именованный необязательный аргумент.


### 16.3. Основные возможности `argparse`

При `add_argument()` часто используют:

- `type=int` — преобразовать значение в число;
- `default=...` — значение по умолчанию;
- `required=True` — сделать именованный аргумент обязательным;
- `help="..."` — текст для справки;
- `choices=[...]` — разрешённые значения;
- `action="store_true"` — логический флаг.

Например:


In [ ]:
# Этот пример предназначен для запуска как отдельный .py-файл:
#
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("filename")
parser.add_argument("--encoding", default="utf-8")
parser.add_argument("--verbose", action="store_true")

args = parser.parse_args()

if args.verbose:
    print("Открываем:", args.filename)

with open(args.filename, encoding=args.encoding) as file:
    print(file.read())


## 17. ZIP-архивы: модуль `zipfile`

### Основные классы и методы

- `ZipFile(path, "r")` — открыть архив;
- `ZipFile(path, "w")` — создать новый;
- `ZipFile(path, "a")` — добавить данные;
- `.write(filename)` — добавить файл;
- `.writestr(name, data)` — записать данные напрямую;
- `.namelist()` — имена файлов;
- `.infolist()` — подробная информация;
- `.open(name)` — открыть файл внутри архива;
- `.read(name)` — прочитать файл;
- `.extract(name, path)` — извлечь один файл;
- `.extractall(path)` — извлечь всё;
- `.getinfo(name)` — информация о файле;
- `ZIP_DEFLATED` — ZIP-сжатие.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

with ZipFile("lecture_example.zip", "w", compression=ZIP_DEFLATED) as archive:
    archive.writestr("example.txt", "Привет из ZIP!")

with ZipFile("lecture_example.zip", "r") as archive:
    print(archive.namelist())
    print(archive.read("example.txt").decode("utf-8"))


### Безопасность при распаковке

Распаковывать неизвестные архивы без проверки нежелательно: внутри могут быть неожиданные пути или очень большой объём данных.

Для учебных примеров используйте собственные или доверенные архивы.
